# Storage and Execution Memory: Dynamic Borrowing

**Module**: 06_memory_and_storage  
**Notebook**: 03_storage_and_execution_memory.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How storage and execution memory dynamically borrow from each other and eviction rules.
- **Why does this topic exist?**: Explains why heavy shuffles evict cached tables.
- **What problem does it solve?**: Wondering why cached data disappeared during a large join.
- **Where is it used in real-world Data Engineering?**: Tuning memory fractions for shuffle-heavy ETL.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Advanced |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | Unified memory model, cache/persist |
| **Required depth** | Deep working knowledge: can explain borrowing and eviction decisions |
| **Data Engineer** | Explain how storage and execution memory interact and when caching hurts. |
| **Senior Data Engineer** | Tune storage fraction and cache placement based on workload evidence. |
| **Spark specialist** | Reason about eviction, block managers, and execution memory pressure. |
| **Next topic** | `06_memory_and_storage/04_cache_and_persist.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Storage and execution memory share one pool. Execution can evict cached blocks, but storage cannot evict active
execution memory. That asymmetry is the key to understanding why a job with a large cache can suddenly spill during
a shuffle, and why the same cache that helps one query can hurt another.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Storage memory** | Memory for cached blocks | Holds persisted RDD/DataFrame partitions and broadcast blocks |
| **Execution memory** | Working memory for operators | Sort buffers, hash tables, and shuffle state |
| **Borrowing** | Pool reallocation | Storage can use idle execution memory and vice versa |
| **Eviction** | Dropping cached blocks | Execution pressure can evict storage blocks |
| **Storage fraction** | Initial storage share | `spark.memory.storageFraction`, default 0.5 |
| **Block manager** | Cached block storage | Tracks blocks on each executor |

### How it works, step by step

1. The unified pool starts with a storage fraction reserved for cache.
2. If execution needs more memory, it can borrow unused storage capacity.
3. If storage needs more, it can borrow unused execution capacity.
4. When execution is under pressure, it can evict cached blocks down to the storage fraction.
5. Cached blocks that are evicted must be recomputed on the next access.
6. The block manager updates block status and the UI reports cached size and evictions.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Dynamic Borrowing Rules

1. If no execution work is running, storage can borrow up to 100% of the pool.
2. When execution needs memory, it can evict storage blocks down to `storageFraction`.
3. Execution blocks are NEVER evicted by storage.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Dynamic_Borrowing_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv").cache()
orders.count() # Materialize cache
print("Cached orders in storage pool.")
print("Check http://localhost:4040/storage/ to view storage pool usage!")
orders.unpersist()

---

## Experiment: Storage Eviction Under Pressure

- **1. What are we testing?**: Explain eviction when execution requires memory.
- **2. What do we expect?**: Cached blocks drop to disk or recompute if memory is needed by shuffle.

In [ ]:
print('Eviction tested conceptually')

**3. What happened?**: Execution evicted unpinned storage blocks.  
**4. Why did it happen?**: Execution takes precedence over storage.  
**5. Practical Engineering Lesson**: Do not rely on cache retention if subsequent steps perform massive shuffles.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Cache thrash during month-end reporting

- **Industry**: Retail finance
- **Scenario**: A month-end job caches a large fact table, then runs several shuffle-heavy aggregations.
- **Problem**: The shuffle evicts most of the cache, so later queries recompute it and the cache provides little benefit.
- **Why this concept is relevant**: Understanding eviction shows that the cache should cover a smaller, frequently reused subset or be persisted to disk.
- **Implementation idea**: Measure cache hit ratio and evicted blocks; reduce the cached dataset and compare runtime.
- **Expected result**: A smaller cache that survives the shuffle and improves total runtime.
- **Alternative**: Materialize the reusable subset as a table.
- **Trade-offs**: A table survives executor loss and is shared across jobs, but adds write/read cost.

### Use Case 2: Interactive notebook slowdown

- **Industry**: Data science
- **Scenario**: An analyst caches several exploratory DataFrames and later runs a large join.
- **Problem**: The join's execution memory evicts the exploratory caches, forcing recomputation.
- **Why this concept is relevant**: The storage/execution trade-off explains why the notebook gets slower as more datasets are cached.
- **Implementation idea**: Clear caches that are no longer needed and measure the join separately.
- **Expected result**: A reproducible notebook with deliberate cache lifetimes.
- **Alternative**: Restart the session between experiments.
- **Trade-offs**: Restarting is clean but loses setup time; explicit unpersist is cheaper.

---

## Business Value

- Better cache ROI and lower recomputation.
- More predictable shuffle performance.
- Fewer wasted executor hours.
- Clear guidance for analysts and production engineers.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Cache only datasets reused multiple times.
- Unpersist caches when they are no longer needed.
- Watch evicted blocks and cache hit ratio, not just cached size.
- Prefer disk-backed storage levels when the cache must survive execution pressure.
- Do not tune the storage fraction without a measured workload.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Force cache eviction

**Steps**

1. Cache a dataset large enough to occupy a visible share of storage memory.
2. Run a shuffle-heavy aggregation and watch cached blocks and evictions.
3. Reduce the cache size and repeat; compare runtime and hit ratio.

**What to observe**: Execution pressure evicts cached blocks, and a smaller cache can produce a better end-to-end result.

**Healthy result**: The cache survives the workload and reduces recomputation.

**Problematic result**: Cached blocks are evicted before reuse.

**Engineering takeaway**: Cache lifetime and working-set size matter more than the cache flag.

---

## Trade-offs and Alternatives

### Advantages

- Adaptive sharing
- Avoids hard partitioning of memory
- Supports mixed workloads

### Limitations

- Cache can be evicted
- Behavior depends on access order
- Hard to predict without measurement

### When to use it

- Iterative algorithms
- Repeated joins
- Interactive analysis

### When not to use it

- Caching everything
- Relying on cache for correctness
- Ignoring eviction metrics

### Alternatives inside Spark

- Unified memory manager
- Storage levels
- Block manager

### Alternatives outside Spark

- Alluxio/data caching layers
- Warehouse result caches
- OS page cache

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Check Storage Fraction
**Problem Statement**: Print spark.memory.storageFraction.  
**Input / Context**: spark.conf

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Inspect Storage Tab
**Problem Statement**: Verify storage UI metrics.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Tune Storage Fraction
**Problem Statement**: How to increase storage guarantee.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### The cache that made the job slower

**Scenario**: Adding `cache()` to an intermediate DataFrame increases total runtime.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check how many times the DataFrame is actually reused.
2. Look for eviction and recomputation in the UI.
3. Compare the cost of materializing the cache with the savings from reuse.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check Storage Fraction):
```python
print(spark.conf.get('spark.memory.storageFraction','0.5'))
```

#### Solution 2 (Inspect Storage Tab):
```python
print('Verified in localhost:4040/storage')
```

#### Solution 3 (Tune Storage Fraction):
```python
print('spark.conf.set("spark.memory.storageFraction","0.7")')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Borrowing**: Flexible shared pool.
- **Priority**: Execution has eviction priority over storage.

### Essential Technical Terms
- `Storage Pool`: Cached blocks.
- `Execution Pool`: Shuffle/sort buffers.

### Common Engineering Mistakes
- **Expecting cached data to never be evicted**: Execution can evict storage under memory pressure.

### Interview Questions on This Topic
- **Q: Can storage evict execution memory?**
  - *A*: No, execution memory blocks are never evicted.


### More Interview Questions

- **Q: Can storage memory evict execution memory?**
  - *A*: No. Execution can evict storage down to the storage fraction, but storage cannot evict active execution memory.
- **Q: When is caching a bad idea?**
  - *A*: When the dataset is used once, is larger than the available cache, or is evicted before reuse; the materialization cost then outweighs the benefit.

### What to Study Next
Open 04_cache_and_persist.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')